# Lightweight CNN-Based Disaster Detection Framework
## End-to-End Training Notebook

This notebook walks through the complete pipeline for training, evaluating, and
optimising a MobileNetV2-based image classifier that detects disaster scenes.

### Pipeline Steps
1. **Import Libraries** – Load all required packages.
2. **Configuration** – Set hyperparameters and file paths.
3. **Data Loading & Preprocessing** – Build augmented data generators.
4. **Visualise Sample Images** – Sanity-check the loaded data.
5. **Build the Model** – MobileNetV2 transfer-learning architecture.
6. **Compile & Train** – Run the training loop with callbacks.
7. **Plot Training History** – Accuracy and loss curves.
8. **Model Evaluation** – Metrics, confusion matrix, classification report.
9. **Model Optimisation** – Convert to TFLite and benchmark inference.
10. **Single-Image Prediction** – End-to-end test on a new image.

## 1. Import Libraries

In [ ]:
# Standard library
import os
import time

# Numerical and data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    Input,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
)

# Scikit-learn evaluation utilities
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs available     : {tf.config.list_physical_devices('GPU')}")

## 2. Configuration and Hyperparameters

In [ ]:
# ── Image dimensions ────────────────────────────────────────────
IMG_HEIGHT = 224          # MobileNetV2 default input height
IMG_WIDTH  = 224          # MobileNetV2 default input width
IMG_SIZE   = (IMG_HEIGHT, IMG_WIDTH)

# ── Training hyperparameters ───────────────────────────────────────────
BATCH_SIZE    = 32        # Mini-batch size for gradient updates
EPOCHS        = 20        # Maximum number of training epochs
LEARNING_RATE = 0.0001    # Adam optimizer initial learning rate
NUM_CLASSES   = 5         # fire, flood, landslide, earthquake_damage, normal

# ── Data augmentation parameters ─────────────────────────────────────────
AUG_ROTATION_RANGE   = 30   # Max rotation in degrees
AUG_SHIFT_RANGE      = 0.2  # Width and height shift fraction
AUG_SHEAR_RANGE      = 0.2  # Shear transformation fraction
AUG_ZOOM_RANGE       = 0.2  # Zoom fraction
VALIDATION_SPLIT     = 0.2  # Fraction of data reserved for validation

# ── Model architecture parameters ────────────────────────────────────────
DENSE_UNITS_1  = 128  # Neurons in first fully-connected layer
DENSE_UNITS_2  = 64   # Neurons in second fully-connected layer
DROPOUT_RATE_1 = 0.3  # Dropout probability after first dense layer
DROPOUT_RATE_2 = 0.2  # Dropout probability after second dense layer

# ── Paths ─────────────────────────────────────────────────────────────────────────────
DATA_DIR         = '../data'                             # Root dataset directory
MODEL_SAVE_PATH  = '../models/disaster_model.h5'         # Keras model output
TFLITE_SAVE_PATH = '../models/disaster_model.tflite'     # TFLite model output

# ── Class names (must match subfolder names in DATA_DIR) ─────────────────────
CLASS_NAMES = [
    'earthquake_damage',
    'fire',
    'flood',
    'landslide',
    'normal',
]

print("Configuration loaded successfully.")
print(f"  Classes    : {CLASS_NAMES}")
print(f"  Image size : {IMG_SIZE}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Epochs     : {EPOCHS}")
print(f"  LR         : {LEARNING_RATE}")
print(f"  Dropout    : {DROPOUT_RATE_1} / {DROPOUT_RATE_2}")

## 3. Data Loading and Preprocessing

We use **`ImageDataGenerator`** for:
- **Training set**: heavy augmentation to improve generalisation.
- **Validation/test set**: rescaling only (no augmentation) to get unbiased metrics.

An 80/20 stratified split is applied via `validation_split=0.2`.

In [ ]:
# ── Training generator with augmentation ───────────────────────────────────────
# Augmentation makes the model robust to real-world image variations:
#  • AUG_ROTATION_RANGE   – handles tilted camera angles
#  • AUG_SHIFT_RANGE      – handles off-center subjects
#  • AUG_SHEAR/ZOOM_RANGE – handles perspective distortion
#  • horizontal_flip      – doubles the effective dataset size
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,                      # Normalise pixel values to [0, 1]
    rotation_range=AUG_ROTATION_RANGE,         # Random rotation
    width_shift_range=AUG_SHIFT_RANGE,         # Horizontal shift
    height_shift_range=AUG_SHIFT_RANGE,        # Vertical shift
    shear_range=AUG_SHEAR_RANGE,               # Shear transformation
    zoom_range=AUG_ZOOM_RANGE,                 # Zoom in/out
    horizontal_flip=True,                      # Random left-right flip
    fill_mode='nearest',                        # Fill newly created pixels
    validation_split=VALIDATION_SPLIT,         # Reserve fraction for validation
)

# ── Validation/test generator WITHOUT augmentation ───────────────────────────
test_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=VALIDATION_SPLIT,
)

# ── Build generators from directory ───────────────────────────────────────────
train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',     # One-hot encoded labels for multi-class
    subset='training',
    shuffle=True,
    seed=42,
)

validation_generator = test_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,                # Keep order for evaluation
    seed=42,
)

# ── Dataset summary ───────────────────────────────────────────────────────────────
print("Dataset summary")
print("-" * 40)
print(f"  Training samples   : {train_generator.samples}")
print(f"  Validation samples : {validation_generator.samples}")
print(f"  Class indices      : {train_generator.class_indices}")
print(f"  Steps per epoch    : {len(train_generator)}")

## 4. Visualise Sample Images

In [ ]:
def plot_sample_images(generator, class_names, rows=3, cols=5):
    """Display a grid of sample images from the training generator.

    Parameters
    ----------
    generator : DirectoryIterator
        Keras image data generator (already fitted to a directory).
    class_names : list[str]
        Ordered list of class label strings.
    rows : int
        Number of grid rows.  Default 3.
    cols : int
        Number of grid columns.  Default 5.
    """
    # Invert the class_indices dict so we can map index → name
    idx_to_class = {v: k for k, v in generator.class_indices.items()}

    # Fetch one batch of images and labels
    images, labels = next(generator)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()

    for i in range(rows * cols):
        if i >= len(images):
            axes[i].axis('off')
            continue
        axes[i].imshow(images[i])                         # Already in [0, 1]
        class_idx = np.argmax(labels[i])
        axes[i].set_title(idx_to_class.get(class_idx, '?'), fontsize=10)
        axes[i].axis('off')

    plt.suptitle('Sample Training Images', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


plot_sample_images(train_generator, CLASS_NAMES)

## 5. Build the Model (MobileNetV2 Transfer Learning)

**Architecture rationale:**

| Layer | Output shape | Parameters |
|-------|-------------|------------|
| Input | (224, 224, 3) | 0 |
| MobileNetV2 (frozen) | (7, 7, 1280) | ~2.2 M (frozen) |
| GlobalAveragePooling2D | (1280,) | 0 |
| Dense(128, ReLU) | (128,) | 163 968 |
| Dropout(0.3) | (128,) | 0 |
| Dense(64, ReLU) | (64,) | 8 256 |
| Dropout(0.2) | (64,) | 0 |
| Dense(5, Softmax) | (5,) | 325 |

MobileNetV2 uses depthwise separable convolutions, reducing FLOPS by ~8–9× vs.
standard convolutions while retaining strong ImageNet feature representations.

In [ ]:
def build_disaster_model(num_classes=NUM_CLASSES, input_shape=(224, 224, 3)):
    """Build and return the MobileNetV2-based disaster detection model.

    The MobileNetV2 base is loaded with ImageNet weights and all its layers are
    frozen (``trainable=False``).  Only the custom classification head is
    trained, which drastically reduces training time and prevents overfitting
    on small datasets.

    Parameters
    ----------
    num_classes : int
        Number of output classes.  Default 5.
    input_shape : tuple[int, int, int]
        (H, W, C) input shape.  Default (224, 224, 3).

    Returns
    -------
    tf.keras.Model
        Compiled-ready model with frozen MobileNetV2 backbone.
    """
    # ── Base model ────────────────────────────────────────────────────────────
    # include_top=False removes the original ImageNet classification head
    # so we can attach our own head tailored for 5 disaster classes.
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
    )

    # Freeze all base layers – we only want to train the custom head.
    # This speeds up training and avoids destroying the learned features.
    base_model.trainable = False

    # ── Custom classification head ────────────────────────────────────────────
    inputs = Input(shape=input_shape, name='input_image')

    # Pass inputs through the frozen base model (inference mode)
    x = base_model(inputs, training=False)

    # Global Average Pooling converts the (7,7,1280) feature map to (1280,),
    # collapsing spatial dimensions while preserving channel information.
    x = GlobalAveragePooling2D(name='global_avg_pool')(x)

    # First fully-connected layer
    x = Dense(DENSE_UNITS_1, activation='relu', name='dense_128')(x)
    x = Dropout(DROPOUT_RATE_1, name='dropout_0.3')(x)  # 30% dropout to prevent overfitting

    # Second fully-connected layer
    x = Dense(DENSE_UNITS_2, activation='relu', name='dense_64')(x)
    x = Dropout(DROPOUT_RATE_2, name='dropout_0.2')(x)  # 20% dropout

    # Output layer: softmax gives a probability distribution over 5 classes
    outputs = Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inputs, outputs=outputs, name='disaster_detector')

    # ── Summary ───────────────────────────────────────────────────────────────
    model.summary()

    trainable_params     = sum(np.prod(v.shape) for v in model.trainable_weights)
    non_trainable_params = sum(np.prod(v.shape) for v in model.non_trainable_weights)
    print(f"\nTrainable parameters     : {trainable_params:,}")
    print(f"Non-trainable parameters : {non_trainable_params:,}")

    return model


model = build_disaster_model()

## 6. Compile and Train the Model

In [ ]:
# ── Compile ───────────────────────────────────────────────────────────────────
# Adam optimizer with a small learning rate is well-suited for fine-tuning:
#  • categorical_crossentropy – standard loss for multi-class classification
#  • accuracy                 – easy-to-interpret monitoring metric
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

# ── Callbacks ─────────────────────────────────────────────────────────────────
# EarlyStopping: halt training if val_loss does not improve for 5 epochs,
#                and restore the best observed weights.
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

# ReduceLROnPlateau: halve the learning rate if val_loss stalls for 3 epochs,
#                    helping escape shallow local minima.
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-7,
)

# ModelCheckpoint: save the model whenever val_accuracy improves.
checkpoint = ModelCheckpoint(
    filepath=MODEL_SAVE_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1,
)

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

# ── Train ─────────────────────────────────────────────────────────────────────
print("Starting training…")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1,
)

print("\nTraining complete.")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")

## 7. Plot Training History

In [ ]:
def plot_training_history(history, save_path='../models/training_history.png'):
    """Plot and save training / validation accuracy and loss curves.

    Parameters
    ----------
    history : tf.keras.callbacks.History
        Object returned by ``model.fit()``.
    save_path : str
        File path to save the figure.  Default '../models/training_history.png'.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs_range = range(1, len(history.history['accuracy']) + 1)

    # ── Accuracy ──────────────────────────────────────────────────────────────
    axes[0].plot(epochs_range, history.history['accuracy'],
                 label='Train', marker='o', linewidth=2)
    axes[0].plot(epochs_range, history.history['val_accuracy'],
                 label='Validation', marker='s', linewidth=2)
    axes[0].set_title('Model Accuracy', fontsize=14)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # ── Loss ──────────────────────────────────────────────────────────────────
    axes[1].plot(epochs_range, history.history['loss'],
                 label='Train', marker='o', linewidth=2, color='orange')
    axes[1].plot(epochs_range, history.history['val_loss'],
                 label='Validation', marker='s', linewidth=2, color='red')
    axes[1].set_title('Model Loss', fontsize=14)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Training History', fontsize=16)
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure saved to: {save_path}")


plot_training_history(history)

## 8. Model Evaluation

In [ ]:
# ── Load best model from disk ─────────────────────────────────────────────────
best_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
print(f"Loaded best model from: {MODEL_SAVE_PATH}")

# ── Generate predictions on the validation set ────────────────────────────────
# Reset generator to ensure we start from the beginning
validation_generator.reset()

y_pred_probs = best_model.predict(validation_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)          # Predicted class indices
y_true = validation_generator.classes             # Ground-truth class indices

# ── Compute metrics ───────────────────────────────────────────────────────────
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

# ── Formatted results table ───────────────────────────────────────────────────
print("\n" + "=" * 45)
print(" Evaluation Results")
print("=" * 45)
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1-Score (weighted)'],
    'Score':  [f'{acc:.4f}', f'{prec:.4f}', f'{rec:.4f}', f'{f1:.4f}'],
})
print(results_df.to_string(index=False))
print("=" * 45)

# ── Per-class classification report ──────────────────────────────────────────
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
def plot_confusion_matrix(
    y_true,
    y_pred,
    class_names,
    save_path='../models/confusion_matrix.png',
):
    """Plot and save a normalised confusion matrix as a seaborn heatmap.

    Each cell shows the fraction of true-class samples predicted as each class,
    making it easy to identify per-class error patterns.

    Parameters
    ----------
    y_true : array-like
        Ground-truth integer class labels.
    y_pred : array-like
        Predicted integer class labels.
    class_names : list[str]
        Human-readable class name for each index.
    save_path : str
        Destination file path for the saved figure.
    """
    cm = confusion_matrix(y_true, y_pred)
    # Normalise row-wise to get recall per class
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=0.5,
    )
    plt.title('Normalised Confusion Matrix', fontsize=14)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Confusion matrix saved to: {save_path}")


plot_confusion_matrix(y_true, y_pred, CLASS_NAMES)

## 9. Model Optimisation for Resource-Constrained Environments

We convert the Keras model to **TensorFlow Lite** with post-training dynamic
range quantisation (`Optimize.DEFAULT`).  This:
- Quantises weights from float32 → int8 at inference time
- Reduces model size by ~75%
- Speeds up inference ~3–4× on ARM/mobile CPUs
- Requires no calibration data

In [ ]:
# ── Convert to TFLite ─────────────────────────────────────────────────────────
print("Converting Keras model to TFLite with dynamic range quantisation…")

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Dynamic range quantisation

tflite_model = converter.convert()

os.makedirs(os.path.dirname(TFLITE_SAVE_PATH), exist_ok=True)
with open(TFLITE_SAVE_PATH, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model saved to: {TFLITE_SAVE_PATH}")

# ── Size comparison ───────────────────────────────────────────────────────────
keras_size_mb  = os.path.getsize(MODEL_SAVE_PATH) / (1024 ** 2)
tflite_size_mb = os.path.getsize(TFLITE_SAVE_PATH) / (1024 ** 2)
compression    = keras_size_mb / tflite_size_mb
reduction_pct  = (1.0 - tflite_size_mb / keras_size_mb) * 100.0

print("\nModel Size Comparison")
print("-" * 40)
print(f"  Keras (.h5)   : {keras_size_mb:.2f} MB")
print(f"  TFLite        : {tflite_size_mb:.2f} MB")
print(f"  Compression   : {compression:.1f}×")
print(f"  Size reduction: {reduction_pct:.1f}%")

In [ ]:
# ── Benchmark inference speed ─────────────────────────────────────────────────
# A synthetic input image is used so we can benchmark without real data.

BENCH_RUNS  = 50
WARMUP_RUNS = 5

sample_input = np.random.rand(1, IMG_HEIGHT, IMG_WIDTH, 3).astype(np.float32)


def measure_keras_inference(model, input_data, n_runs=BENCH_RUNS, n_warmup=WARMUP_RUNS):
    """Measure average Keras model inference time over ``n_runs`` runs.

    Warm-up runs are performed first to allow TF to JIT-compile the graph and
    load weights into GPU/CPU cache, giving a realistic steady-state estimate.

    Parameters
    ----------
    model : tf.keras.Model
    input_data : np.ndarray, shape (1, 224, 224, 3)
    n_runs : int  – number of timed inference calls
    n_warmup : int – number of warm-up calls (not timed)

    Returns
    -------
    float  – average inference time in milliseconds
    """
    for _ in range(n_warmup):
        model.predict(input_data, verbose=0)

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.predict(input_data, verbose=0)
        times.append((time.perf_counter() - t0) * 1000.0)

    return float(np.mean(times))


def measure_tflite_inference(
    tflite_path, input_data, n_runs=BENCH_RUNS, n_warmup=WARMUP_RUNS
):
    """Measure average TFLite interpreter inference time over ``n_runs`` runs.

    Parameters
    ----------
    tflite_path : str  – path to the .tflite file
    input_data : np.ndarray, shape (1, 224, 224, 3)
    n_runs : int
    n_warmup : int

    Returns
    -------
    float  – average inference time in milliseconds
    """
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    inp  = interpreter.get_input_details()[0]
    out  = interpreter.get_output_details()[0]

    cast_input = input_data.astype(inp['dtype'])

    for _ in range(n_warmup):
        interpreter.set_tensor(inp['index'], cast_input)
        interpreter.invoke()

    times = []
    for _ in range(n_runs):
        interpreter.set_tensor(inp['index'], cast_input)
        t0 = time.perf_counter()
        interpreter.invoke()
        times.append((time.perf_counter() - t0) * 1000.0)

    return float(np.mean(times))


# Run benchmarks
keras_ms  = measure_keras_inference(best_model, sample_input)
tflite_ms = measure_tflite_inference(TFLITE_SAVE_PATH, sample_input)
speedup   = keras_ms / tflite_ms if tflite_ms > 0 else float('inf')

# Rough RAM estimates (weights + activations)
keras_ram_mb  = keras_size_mb * 4   # float32 activations roughly 4× model size
tflite_ram_mb = tflite_size_mb * 2  # int8 model uses less RAM

# Formatted comparison table
print("\nInference Benchmark (CPU, single image)")
print("=" * 60)
print(f"{'Metric':<30} {'Keras (.h5)':>14} {'TFLite':>14}")
print("-" * 60)
print(f"{'Model size (MB)':<30} {keras_size_mb:>14.2f} {tflite_size_mb:>14.2f}")
print(f"{'Avg inference time (ms)':<30} {keras_ms:>14.2f} {tflite_ms:>14.2f}")
print(f"{'Speedup':<30} {'1.0×':>14} {speedup:>13.1f}×")
print(f"{'Est. RAM usage (MB)':<30} {keras_ram_mb:>14.1f} {tflite_ram_mb:>14.1f}")
print("=" * 60)

## 10. Test Prediction on a Single Image

In [ ]:
def predict_single_image(image_path, model, class_names=CLASS_NAMES, img_size=IMG_SIZE):
    """Classify a single image and display the result with a confidence bar chart.

    Parameters
    ----------
    image_path : str
        Absolute or relative path to the image file.
    model : tf.keras.Model
        Trained Keras model.
    class_names : list[str]
        Ordered list of class label strings.
    img_size : tuple[int, int]
        (H, W) to which the image is resized before inference.
    """
    from tensorflow.keras.preprocessing import image as keras_image  # noqa: PLC0415

    # ── Load and preprocess ────────────────────────────────────────────────────
    img = keras_image.load_img(image_path, target_size=img_size)
    img_array = keras_image.img_to_array(img) / 255.0  # Normalise to [0, 1]
    img_array = np.expand_dims(img_array, axis=0)       # Add batch dimension

    # ── Predict ────────────────────────────────────────────────────────────────
    predictions = model.predict(img_array, verbose=0)[0]  # shape: (5,)
    pred_idx    = int(np.argmax(predictions))
    pred_class  = class_names[pred_idx]
    confidence  = predictions[pred_idx] * 100.0

    # ── Visualise ─────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: original image with prediction overlay
    axes[0].imshow(img)
    axes[0].set_title(
        f'Prediction: {pred_class}\nConfidence: {confidence:.1f}%',
        fontsize=13,
        fontweight='bold',
    )
    axes[0].axis('off')

    # Right: bar chart of all class probabilities
    colours = ['#e74c3c' if i == pred_idx else '#3498db' for i in range(len(class_names))]
    axes[1].barh(class_names, predictions * 100.0, color=colours)
    axes[1].set_xlim(0, 100)
    axes[1].set_xlabel('Probability (%)')
    axes[1].set_title('Class Probabilities', fontsize=13)
    for i, prob in enumerate(predictions):
        axes[1].text(prob * 100.0 + 1.0, i, f'{prob * 100.0:.1f}%', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    print(f"\n→ Predicted class : {pred_class}")
    print(f"  Confidence      : {confidence:.2f}%")


# ── Example usage (uncomment and provide a valid path) ────────────────────────
# predict_single_image('../data/fire/sample.jpg', best_model)

## Summary

| # | Step | Status |
|---|------|--------|
| 1 | Import libraries | ✅ |
| 2 | Configuration | ✅ |
| 3 | Data loading & preprocessing | ✅ |
| 4 | Visualise sample images | ✅ |
| 5 | Build MobileNetV2 model | ✅ |
| 6 | Compile & train | ✅ |
| 7 | Plot training history | ✅ |
| 8 | Evaluate (metrics + confusion matrix) | ✅ |
| 9 | Convert to TFLite & benchmark | ✅ |
| 10 | Single-image prediction | ✅ |

### Next Steps

- **Fine-tune the base model**: unfreeze the top layers of MobileNetV2 and run a second training phase with a very small learning rate (e.g. 1e-5) to squeeze out extra accuracy.
- **Edge deployment**: copy `disaster_model.tflite` to a Raspberry Pi or Android device and integrate with the on-device TFLite runtime.
- **Real-world dataset**: collect or augment with additional disaster images (e.g. from AIDER, MEDIC, or Flickr disaster datasets).
- **Streamlit app**: launch `app/app.py` for an interactive web demo.